<a href="https://colab.research.google.com/github/francisdick/detecteur-langue-fon/blob/main/03_regles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 3 : détecteur à base de règles (score de référence)

| Version | Score global | Fon avec tons | Fon WhatsApp | Français | Anglais |
|---|---|---|---|---|---|
| Règles v1 | 63,1 % | 86,7 % | 0 % | 100 % | 46,0 % |
| Règles v2 | **82,2 %** | 98,7 % | 40,7 % | 87,7 % | 89,3 % |

**Score de référence retenu : règles v2, 82,2 %.** Le modèle de la phase 4 devra le dépasser, en particulier sur le fon WhatsApp (40,7 %).

Cellule 1 : connexion et chargement

In [15]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
DOSSIER = "/content/drive/MyDrive/FRANCIS DICK CLAUDE IA/Projet NLP - Détecteur de Langue Fon/"
df = pd.read_csv(DOSSIER + "phrases_propres.csv")
print(df.shape)
print(df["langue"].value_counts())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(900, 3)
langue
fon    300
en     300
fr     300
Name: count, dtype: int64


In [16]:
# Toutes les sources "tatoeba_XXXX" deviennent simplement "tatoeba"
df["corpus"] = df["source"].where(~df["source"].str.startswith("tatoeba"), "tatoeba")
df.groupby(["langue", "corpus"]).size()

langue  corpus                  
en      tatoeba                     300
fon     zenodo_4432712              150
        zenodo_4432712_simplifie    150
fr      tatoeba                     300
dtype: int64

Cellule 2 : la fonction de nettoyage

In [17]:
import unicodedata, re

def normaliser(texte):
    """Minuscules + suppression de la ponctuation, sans toucher aux tons du fon."""
    texte = unicodedata.normalize("NFC", str(texte)).lower()
    texte = "".join(" " if unicodedata.category(c).startswith("P") else c for c in texte)
    return re.sub(r"\s+", " ", texte).strip()

print(normaliser("Where are you?"))   # where are you
print(normaliser("A fɔ́n ganji à ?"))  # a fɔ́n ganji à

where are you
a fɔ́n ganji à


Cellule 3 : les règles version 1 (celles du guide)

In [18]:
LETTRES_FON = set("ɔɛɖ")
MOTS_EN_V1 = {"the", "is", "and", "you"}

def regles_v1(phrase):
    p = normaliser(phrase)
    if LETTRES_FON & set(p):           # au moins une lettre propre au fon
        return "fon"
    if MOTS_EN_V1 & set(p.split()):    # au moins un mot anglais très courant
        return "en"
    return "fr"                        # sinon, on parie sur le français

df["pred_v1"] = df["texte"].apply(regles_v1)

Cellule 4 : mesurer le score

In [19]:
df["ok_v1"] = df["pred_v1"] == df["langue"]
print(f"Score v1 : {df['ok_v1'].mean():.1%}")

# Tableau de confusion : les lignes = vraie langue, les colonnes = langue prédite
pd.crosstab(df["langue"], df["pred_v1"], rownames=["vraie"], colnames=["prédite"])

Score v1 : 63.1%


prédite,en,fon,fr
vraie,,,
en,138,0,162
fon,0,130,170
fr,0,0,300


Cellule 5 : le score par source

In [20]:
df.groupby(["langue", "corpus"])["ok_v1"].agg(["count", "mean"])

count      mean
langue corpus                                   
en     tatoeba                     300  0.460000
fon    zenodo_4432712              150  0.866667
       zenodo_4432712_simplifie    150  0.000000
fr     tatoeba                     300  1.000000

Cellule 6 : examiner les erreurs

In [21]:
erreurs_v1 = df[~df["ok_v1"]]
print(len(erreurs_v1), "erreurs")
erreurs_v1[["texte", "langue", "pred_v1", "source"]].sample(min(10, len(erreurs_v1)), random_state=42)

332 erreurs


,texte,langue,pred_v1,source
83,adu ton bi wo,fon,fr,zenodo_4432712_simplifie
856,I failed my chemistry exam.,en,fr,tatoeba_13210047
218,mi je kpoli,fon,fr,zenodo_4432712_simplifie
529,Tom looks like a hero to me.,en,fr,tatoeba_5205183
179,a du su nu mi desu,fon,fr,zenodo_4432712_simplifie
455,azan dagbe,fon,fr,zenodo_4432712_simplifie
758,xo de nu me,fon,fr,zenodo_4432712_simplifie
477,ce kwe nu mi,fon,fr,zenodo_4432712_simplifie
577,I should leave early.,en,fr,tatoeba_8848619
534,e no da ayihun debu a,fon,fr,zenodo_4432712_simplifie


### Hypothèses sur 10 erreurs de la v1

| Phrase | Vraie | Prédite | Pourquoi les règles se trompent |
|---|---|---|---|
| adu ton bi wo | fon | fr | Écrite sans ɔ, ɛ ni ɖ → tombe dans « sinon, français » |
| I failed my chemistry exam. | en | fr | Aucun des 4 mots anglais de la liste (the, is, and, you) |
| mi je kpoli | fon | fr | Sans lettre spéciale ; « je » ressemble en plus à du français |
| Tom looks like a hero to me. | en | fr | Aucun des 4 mots anglais de la liste |
| a du su nu mi desu | fon | fr | Sans lettre spéciale (style WhatsApp) |
| azan dagbe | fon | fr | Phrase très courte, sans lettre spéciale |
| xo de nu me | fon | fr | Sans lettre spéciale ; « de » ressemble à du français |
| ce kwe nu mi | fon | fr | Sans lettre spéciale ; « ce » ressemble à du français |
| I should leave early. | en | fr | Aucun des 4 mots anglais de la liste |
| e no da ayihun debu a | fon | fr | Sans lettre spéciale (style WhatsApp) |

**Bilan :** 7 erreurs sur 10 sont du fon WhatsApp, 3 sont de l'anglais sans les 4 mots de la liste.

Cellule 7 : les règles version 2, à toi de jouer

In [22]:
MOTS = {
    "fr":  {"je", "de", "la", "le", "les", "est", "pas", "et", "un", "une",
            "vous", "il", "que", "ce", "au", "du", "tu", "nous", "en", "des"},
    "en":  {"the", "is", "and", "you", "i", "to", "it", "of", "in", "my",
            "this", "what", "he", "are", "was", "do", "not", "have", "we", "she"},
    "fon": {
        "sin", "wema", "agbaza", "nukun", "kabo", "xowe", "agbade", "hwevi", "ayikungban",
        "kpevi", "koklo", "fofo", "zinkpo", "egbe", "zanzan", "zozo"
    },
}

def regles_v2(phrase):
    p = normaliser(phrase)
    if LETTRES_FON & set(p):
        return "fon"
    mots = p.split()
    scores = {langue: sum(m in liste for m in mots) for langue, liste in MOTS.items()}
    meilleure = max(scores, key=scores.get)
    return meilleure if scores[meilleure] > 0 else "fon"   # aucun mot reconnu → fon

df["pred_v2"] = df["texte"].apply(regles_v2)
df["ok_v2"] = df["pred_v2"] == df["langue"]
print(f"Score v1 : {df['ok_v1'].mean():.1%}  |  Score v2 : {df['ok_v2'].mean():.1%}")
pd.crosstab(df["langue"], df["pred_v2"], rownames=["vraie"], colnames=["prédite"])

Score v1 : 63.1%  |  Score v2 : 82.2%


prédite,en,fon,fr
vraie,,,
en,268,32,0
fon,43,209,48
fr,0,37,263


Cellule 8 : comparer v1 et v2 par source

In [23]:
df.groupby(["langue", "corpus"])[["ok_v1", "ok_v2"]].mean().round(3)

ok_v1  ok_v2
langue corpus                                
en     tatoeba                   0.460  0.893
fon    zenodo_4432712            0.867  0.987
       zenodo_4432712_simplifie  0.000  0.407
fr     tatoeba                   1.000  0.877

Cellule 9 : sauvegarder la fonction pour la phase 4

In [24]:
print(f"SCORE DE RÉFÉRENCE v1 : {df['ok_v1'].mean():.1%}")
print(f"SCORE DE RÉFÉRENCE v2 : {df['ok_v2'].mean():.1%}")

SCORE DE RÉFÉRENCE v1 : 63.1%
SCORE DE RÉFÉRENCE v2 : 82.2%


## Limites des règles

1. **Fon sans lettres spéciales** : la v2 reconnaît 98,7 % du fon écrit avec ɔ, ɛ et ɖ, mais seulement 40,7 % du fon écrit en style WhatsApp.
2. **Mots ambigus** : sans tons, des mots fon comme `do`, `we`, `un`, `le` ou `de` s'écrivent exactement comme des mots anglais ou français. Les 91 phrases fon mal classées par la v2 (43 prises pour de l'anglais, 48 pour du français) contiennent toutes au moins un mot des listes française ou anglaise.
3. **Listes de mots trop étroites** : ma liste fon (des noms comme *koklo* ou *hwevi*) ne s'est déclenchée que sur 14 phrases. Il faudrait des centaines de mots ajoutés à la main pour couvrir la langue.
4. **Pari par défaut** : une phrase sans mot connu est classée en fon, ce qui produit 69 erreurs sur des phrases françaises et anglaises. Avec « français » par défaut, le score tombe à 77,9 %.
5. **Compromis entre langues** : améliorer la détection du fon et de l'anglais a fait baisser le français de 100 % à 87,7 %.

**Conclusion :** un mot isolé ne suffit pas à reconnaître une langue, il faut tenir compte de ce qui l'entoure. C'est ce que le modèle de la phase 4 apprendra avec les n-grammes de caractères.